In [2]:
import numpy as np
import pandas as pd
from scapy.all import rdpcap, raw
import matplotlib.pyplot as plt
import seaborn as sns



In [20]:
%whos


Variable      Type        Data/Info
-----------------------------------
X_test_mm     memmap      [[[0.5686275  0.5686275  <...> 0.         0.        ]]]
X_train_mm    memmap      [[[0.5686275  0.5686275  <...> 0.02745098 0.52156866]]]
X_val_mm      memmap      [[[0.5686275  0.5686275  <...> 0.4509804  0.6666667 ]]]
Xp            ndarray     263849x64x64: 1080725504 elems, type `float32`, 4322902016 bytes (4122.640625 Mb)
end           int         791548
gc            module      <module 'gc' (built-in)>
np            module      <module 'numpy' from '/ho<...>kages/numpy/__init__.py'>
pd            module      <module 'pandas' from '/h<...>ages/pandas/__init__.py'>
plt           module      <module 'matplotlib.pyplo<...>es/matplotlib/pyplot.py'>
raw           function    <function raw at 0x72633b6d9940>
rdpcap        function    <function rdpcap at 0x726339b1b6a0>
sns           module      <module 'seaborn' from '/<...>ges/seaborn/__init__.py'>
start         int         791548
test_tot

In [19]:
!free -h


               total        used        free      shared  buff/cache   available
Mem:            31Gi       8.8Gi        18Gi       393Mi       5.0Gi        22Gi
Swap:           14Gi       4.1Gi        10Gi


In [18]:
del Xtest1, Xtest2, Xtest3
del Xtrain1, Xtrain2, Xtrain3
del Xval1, Xval2, Xval3
import gc
gc.collect()

0

In [7]:
# --- KONFIGURASI DAN PATH FILE ---
BASE_PATH = '/home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/'
TRAIN_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_19_50_training.pcap'
TRAIN_LABELS_PATH = BASE_PATH + 'y_train.csv'

# Kita definisikan juga path untuk data testing untuk nanti
TEST_PCAP_PATH = BASE_PATH + 'Automotive_Ethernet_with_Attack_original_10_17_20_04_test.pcap'
TEST_LABELS_PATH = BASE_PATH + 'y_test.csv'

print("Langkah 1 Selesai: Library dan path sudah siap.")

Langkah 1 Selesai: Library dan path sudah siap.


In [8]:
# --- MEMUAT LABEL TRAINING ---
print("Memuat file label dari:", TRAIN_LABELS_PATH)
df_labels_train = pd.read_csv(TRAIN_LABELS_PATH , header=None)

# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_train1 = df_labels_train.iloc[:, 1].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_train1)} label.")
print("Contoh 5 label pertama:", labels_train1[:5])
print("Distribusi Label:")
print(df_labels_train.iloc[:, 1].value_counts())

Memuat file label dari: /home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/y_train.csv

Berhasil memuat 1203737 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
1
Normal      954912
Abnormal    248825
Name: count, dtype: int64


In [9]:
# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_train2 = df_labels_train.iloc[:, 2].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_train2)} label.")
print("Contoh 5 label pertama:", labels_train2[:5])
print("Distribusi Label:")
print(df_labels_train.iloc[:, 2].value_counts())


Berhasil memuat 1203737 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
2
Normal    954912
C_D        85466
P_I        64635
F_I        35112
M_F        33765
C_R        29847
Name: count, dtype: int64


In [10]:

def extract_packets(pcap_path, n_bytes=64):
    packets = rdpcap(pcap_path)
    data = []

    for pkt in packets:
        pkt_bytes = raw(pkt)

        if len(pkt_bytes) < n_bytes:
            pkt_bytes = pkt_bytes + bytes(n_bytes - len(pkt_bytes))  # zero padding
        else:
            pkt_bytes = pkt_bytes[:n_bytes]  # truncate

        data.append(np.frombuffer(pkt_bytes, dtype=np.uint8))

    return np.array(data)  # (num_packets, 64)


In [11]:

def build_sequences(X_pkt, y_pkt, window=64, step=1):
    X_seq, y_seq = [], []

    for i in range(0, len(X_pkt) - window + 1, step):
        window_data = X_pkt[i:i+window]
        window_label = y_pkt[i:i+window]

        X_seq.append(window_data.T)  # (64, 64) → (n × l)
        y_seq.append(1 if np.any(window_label == 1) else 0)

    return np.array(X_seq), np.array(y_seq)


training binary class

In [12]:
x_pkt_train=extract_packets(TRAIN_PCAP_PATH)

In [13]:
binary_labels_text   = df_labels_train.iloc[:, 1]
y_label_train  = binary_labels_text.map({'Normal': 0, 'Abnormal': 1}).values

In [14]:

x_seq_train, y_seq_train = build_sequences(x_pkt_train,y_label_train)

In [15]:
x_seq_train.shape, y_seq_train.shape

((1203674, 64, 64), (1203674,))

training multiclass

In [16]:
np.unique(y_seq_train)

array([0, 1])

test binary class

In [17]:
# --- MEMUAT LABEL Testing ---
print("Memuat file label dari:", TEST_LABELS_PATH)
df_labels_test = pd.read_csv(TEST_LABELS_PATH , header=None)

# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_test1 = df_labels_test.iloc[:, 1].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_test1)} label.")
print("Contoh 5 label pertama:", labels_test1[:5])
print("Distribusi Label:")
print(df_labels_test.iloc[:, 1].value_counts())
print(df_labels_test.head())


Memuat file label dari: /home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/y_test.csv

Berhasil memuat 791611 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
1
Normal      660777
Abnormal    130834
Name: count, dtype: int64
   0       1       2
0  1  Normal  Normal
1  2  Normal  Normal
2  3  Normal  Normal
3  4  Normal  Normal
4  5  Normal  Normal


In [18]:
# Mengambil semua nilai dari kolom pertama dan menyimpannya sebagai array numpy
labels_test2 = df_labels_test.iloc[:, 2].values

# --- MEMERIKSA LABEL ---
print(f"\nBerhasil memuat {len(labels_test2)} label.")
print("Contoh 5 label pertama:", labels_test2[:5])
print("Distribusi Label:")
print(df_labels_test.iloc[:, 2].value_counts())


Berhasil memuat 791611 label.
Contoh 5 label pertama: ['Normal' 'Normal' 'Normal' 'Normal' 'Normal']
Distribusi Label:
2
Normal    660777
C_D        41203
C_R        29847
P_I        26013
F_I        16962
M_F        16809
Name: count, dtype: int64


In [19]:
x_pkt_test=extract_packets(TEST_PCAP_PATH)

In [20]:
binary_labels_text   = df_labels_test.iloc[:, 1]
y_label_test = binary_labels_text.map({'Normal': 0, 'Abnormal': 1}).values

In [21]:

x_seq_test, y_seq_test = build_sequences(x_pkt_test, y_label_test)
x_seq_test.shape, y_seq_test.shape

((791548, 64, 64), (791548,))

multiclass

In [21]:
np.savez_compressed(
    "preprocessingW64S1binaryfloat32/tow_ids_binary_preprocessedW64S1v1.npz",
    x_train=x_seq_train,
    y_train=y_seq_train,
    x_test=x_seq_test,
    y_test=y_seq_test
)


NameError: name 'x_seq_train' is not defined

In [7]:
data = np.load("/home/dani/Documents/tugas akhir/data/TOW-IDS/TOW-IDS-20250906T180910Z-1-001/TOW-IDS/Automotive Ethernet Dataset/preprocessingW64S1binaryfloat32/tow_ids_binary_preprocessedW64S1v1.npz")

x_seq_train = data["x_train"]
y_seq_train = data["y_train"]
x_seq_test = data["x_test"]
y_seq_test = data["y_test"]
    
print(x_seq_train.shape, y_seq_train.shape)
print(x_seq_test.shape, y_seq_test.shape)


(1203674, 64, 64) (1203674,)
(791548, 64, 64) (791548,)


split

In [14]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(
    x_seq_train, y_seq_train,
    test_size=0.3,
    stratify=y_seq_train,
    random_state=42
)

normalize

In [25]:
def normalize_minmax(X):
    return X.astype(np.float32) / 255.0


In [28]:
import numpy as np
import os
import gc

def split_Xy_normalize_and_save_npz(
    X,
    y,
    n_parts=85,
    out_dir="train_parts_norm",
    prefix="train"
):
    """
    Split X and y into n_parts, normalize X per part,
    and save each part immediately to disk.
    """
    assert X.shape[0] == y.shape[0], "X and y must have same length"
    os.makedirs(out_dir, exist_ok=True)

    total = X.shape[0]
    indices = np.array_split(np.arange(total), n_parts)

    for i, idx in enumerate(indices):
        print(f"[{i+1}/{n_parts}] Processing part {i+1}...")

        # Load slice
        X_part = X[idx]
        y_part = y[idx]

        # Normalize PER PART
        X_part = normalize_minmax(X_part)

        # Save
        out_path = f"{out_dir}/{prefix}_part_{i+1:03d}.npz"
        np.savez_compressed(
            out_path,
            X=X_part,
            y=y_part
        )

        # Free RAM
        del X_part, y_part
        gc.collect()

        print(f"    Saved → {out_path}")

    print("✅ DONE: split + normalize + save per part")


In [30]:
split_Xy_normalize_and_save_npz(
    x_train,
    y_train,
    n_parts=85,
    out_dir="preprocessingW64S1binaryfloat32/train_parts",
    prefix="train"
)


[1/85] Processing part 1...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_001.npz
[2/85] Processing part 2...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_002.npz
[3/85] Processing part 3...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_003.npz
[4/85] Processing part 4...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_004.npz
[5/85] Processing part 5...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_005.npz
[6/85] Processing part 6...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_006.npz
[7/85] Processing part 7...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_007.npz
[8/85] Processing part 8...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_008.npz
[9/85] Processing part 9...
    Saved → preprocessingW64S1binaryfloat32/train_parts/train_part_009.npz
[10/85] Processing part 10...
    Saved → preprocessingW64S1binaryfloat32

In [36]:
split_Xy_normalize_and_save_npz(
    x_val,
    y_val,
    n_parts=85,
    out_dir="preprocessingW64S1binaryfloat32/val_parts",
    prefix="val"
)


[1/85] Processing part 1...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_001.npz
[2/85] Processing part 2...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_002.npz
[3/85] Processing part 3...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_003.npz
[4/85] Processing part 4...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_004.npz
[5/85] Processing part 5...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_005.npz
[6/85] Processing part 6...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_006.npz
[7/85] Processing part 7...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_007.npz
[8/85] Processing part 8...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_008.npz
[9/85] Processing part 9...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_009.npz
[10/85] Processing part 10...
    Saved → preprocessingW64S1binaryfloat32/val_parts/val_part_010.npz
[11/85] 

In [59]:
np.unique(y_seq_test)

array([0, 1])

In [37]:
split_Xy_normalize_and_save_npz(
    x_seq_test,
    y_seq_test,
    n_parts=85,
    out_dir="preprocessingW64S1binaryfloat32/test_parts",
    prefix="test"
)


[1/85] Processing part 1...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_001.npz
[2/85] Processing part 2...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_002.npz
[3/85] Processing part 3...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_003.npz
[4/85] Processing part 4...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_004.npz
[5/85] Processing part 5...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_005.npz
[6/85] Processing part 6...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_006.npz
[7/85] Processing part 7...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_007.npz
[8/85] Processing part 8...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_008.npz
[9/85] Processing part 9...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_part_009.npz
[10/85] Processing part 10...
    Saved → preprocessingW64S1binaryfloat32/test_parts/test_p

In [66]:
import numpy as np

data = np.load("preprocessingW64S1binaryfloat32/test_parts/test_part_070.npz")

print("Keys :", data.files)

X = data["X"]
y = data["y"]


Keys : ['X', 'y']


In [67]:
print("X shape :", X.shape)
print("X dtype :", X.dtype)

print("y shape :", y.shape)
print("y dtype :", y.dtype)


X shape : (9312, 64, 64)
X dtype : float32
y shape : (9312,)
y dtype : int64


In [68]:
print("X min :", X.min())
print("X max :", X.max())


X min : 0.0
X max : 1.0


In [69]:
import numpy as np
print("Unique labels:", np.unique(y))


Unique labels: [0 1]


In [70]:
print("One sample X[0] shape:", X[0].shape)
print("Label y[0]:", y[0])


One sample X[0] shape: (64, 64)
Label y[0]: 0
